### Purpose
- Season 3 of Kaggle's Playground Series has been awesome so far! In this notebook I would like to analyze the public & private leaderboard behavior in Season 3.

### Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", None)

### Loading the data

In [ ]:
PUBLIC_LB_PATH = "/kaggle/input/playground-series-season-3-leaderboard-data/playground-series-s3e_-publicleaderboard.csv"
PRIVATE_LB_PATH = "/kaggle/input/playground-series-season-3-leaderboard-data/playground-series-s3e_-privateleaderboard.csv"
PLAYGROUND_PATH = "/kaggle/input/playground-series-s3e_/data.csv"
competitions = pd.read_csv("/kaggle/input/meta-kaggle/Competitions.csv")

N_EPISODES = 20

In [ ]:
playground_series = [f'Playground Series - Season 3, Episode {i}' for i in range(1, N_EPISODES + 1)]

competition_eval_metrics = competitions.query("Subtitle in @playground_series")["EvaluationAlgorithmAbbreviation"].values
competition_eval_metrics[4] = "QWK"
competition_eval_metrics[10] = "MSLE"
competition_eval_metrics[11] = "AUC"
competition_eval_metrics = np.append(competition_eval_metrics, "RMSE")
print(competition_eval_metrics)

In [ ]:
def read_competition_data(path):
    
    n_train_rows, n_test_rows, n_columns = [], [], []

    for ep in range(1, N_EPISODES + 1):
        if ep != 15:
            train_df = pd.read_csv(path.replace("_", f"{ep}").replace("data", "train"))
            test_df = pd.read_csv(path.replace("_", f"{ep}").replace("data", "test"))
        else:
            data = pd.read_csv(path.replace("_", f"{ep}"))
            data["missing"] = data["x_e_out [-]"].isna().astype(int)
            train_df = data.query("missing == 0").drop(columns = "missing")
            test_df = data.query("missing == 1").drop(columns = "missing")
            
        n_train_rows.append(train_df.shape[0])
        n_test_rows.append(test_df.shape[0])
        n_columns.append(train_df.shape[1])
            
    full_df = pd.DataFrame({
        "# Train Rows": n_train_rows,
        "# Test Rows": n_test_rows,
        "# Columns": n_columns
    })
    
    return full_df

In [ ]:
size_df = read_competition_data(PLAYGROUND_PATH)

In [ ]:
def read_lb_data(path):
    
    full_df = pd.DataFrame()
    
    for ep in range(1, N_EPISODES + 1):
        episode_df = pd.read_csv(path.replace("_", f"{ep}"))
        episode_df["Leaderboard"] = path.split("-")[-1].split("lead")[0]
        episode_df["Rank"] = episode_df.index + 1
        episode_df["Episode"] = ep
        episode_df["EvaluationMetric"] = competition_eval_metrics[ep - 1]
        if ep > 18: episode_df = episode_df.rename(columns = {"LastSubmissionDate": "SubmissionDate"})
        episode_df["SubmissionDate"] = pd.to_datetime(episode_df["SubmissionDate"])
        full_df = pd.concat([full_df, episode_df], ignore_index = True)
        
    return full_df

In [ ]:
public_df = read_lb_data(PUBLIC_LB_PATH)
private_df = read_lb_data(PRIVATE_LB_PATH)
df = pd.concat([public_df, private_df], ignore_index = True)

### Visualizing the Total Number of Participants

In [ ]:
plot_df = pd.DataFrame({
    "Playground Episode": [ep for ep in range(1, N_EPISODES + 1)],
    "Number of Participants": [len(df[df["Leaderboard"] == "private"].query(f"Episode == {ep}")) - 2 for ep in range(1, N_EPISODES + 1)]
})

fig = px.bar(plot_df, x = "Playground Episode", y = "Number of Participants", text = "Number of Participants",
             color = "Number of Participants", color_continuous_scale = ["#000000", "#2eb82e"],
             width = 1200, height = 600, template = "plotly_dark",
             title = "Total Number of Participants in each Playground Series Episode")

fig.update_traces(marker = dict(line = dict(width = 1.5, color = "#FFFFFF")))
fig.update(layout_coloraxis_showscale=False)
fig.update_layout(xaxis = dict(dtick = 1))
fig.show()

### Visualizing the Shakeup of Leaderboard Ranking between the Public & Private LB for each Episode
***
### What is rf_benchmark.csv?
- For each Playground competition, @inversion (Kaggle staff) builds a simple RandomForest benchmark on the data without any feature engineering. It's a baseline score one should expect to achieve with a basic approach that appears on the public and private leaderboard along with all of the participants. The rank of rf_benchmark.csv used in the plots below is the rank achieved on the private leaderboard.

In [ ]:
rank_difference_stdevs = []
rank_df = pd.DataFrame()

for ep in range(1, N_EPISODES + 1):
    
    plt.figure(figsize = (10, 8))
    plt.style.use("fivethirtyeight")
    
    merged_df = pd.merge(df.query(f"Leaderboard == 'private' and Episode == {ep}").reset_index(drop=True), 
                         df.query(f"Leaderboard == 'public' and Episode == {ep}").reset_index(drop=True), 
                         on='TeamId', suffixes=('_private', '_public'))

    merged_df["Rank_difference"] = merged_df["Rank_public"] - merged_df["Rank_private"]

    ax = sns.scatterplot(merged_df, s = 8, marker = "D",
                         x = "Rank_private", y = "Rank_public", hue = "Rank_difference",
                        legend = False)
    
    plt.plot(range(len(merged_df)), color='black', linestyle='--', linewidth=1)
    
    colors = np.where(merged_df["Rank_difference"] < 0, "red", "green")
    ax.collections[0].set_color(colors)
    
    if ep not in [13, 15]:
        rf_benchmark = df.query(f"Episode == {ep} and TeamName == 'rf_benchmark.csv' and Leaderboard == 'private'")["Rank"].values[0]
        plt.axvline(x = rf_benchmark, color = "#0066ff", linestyle = '--', linewidth = 1.5, alpha = 0.75)
        plt.text(
            rf_benchmark - 10,
            merged_df["Rank_public"].max() - 40,
            "rf_benchmark.csv",
            color="#0066ff",
            horizontalalignment='right',
            verticalalignment='center'
        )
    
    rank_difference_stdev = round(merged_df.Rank_difference.std(), 2)
    rank_difference_stdevs.append(rank_difference_stdev)
    
    plt.xlabel("Private LB Rank")
    plt.ylabel("Public LB Rank")
    plt.title(f"Episode: {ep} | stdev of public to private rank difference: {rank_difference_stdev}")
    plt.show()

### Visualizing the Standard Deviation of the Rank Difference between the Public and Private Leaderboard

In [ ]:
plot_df = pd.DataFrame({
    "Playground Episode": [ep for ep in range(1, N_EPISODES + 1)],
    "Stdev of Rank Difference": [round(std, 1) for std in rank_difference_stdevs]
})

fig = px.bar(plot_df, x = "Playground Episode", y = "Stdev of Rank Difference", text = "Stdev of Rank Difference", 
             color = "Stdev of Rank Difference", color_continuous_scale = ["#FFFFFF", "#ff0000"],
             width = 1200, height = 600, template = "plotly_dark",
             title = "Standard Deviation of Rank Difference between the Public and Private Leaderboard")

fig.update_traces(marker = dict(line = dict(width = 1.5, color = "#FFFFFF")))
fig.update(layout_coloraxis_showscale=False)
fig.update_layout(xaxis = dict(dtick = 1))
fig.show()

### Visualizing the Rank Difference between the Public and Private Leaderboard

In [ ]:
merged_df = pd.merge(df.query("Leaderboard == 'private'").reset_index(drop=True), 
                         df.query("Leaderboard == 'public'").reset_index(drop=True), 
                         on='TeamId', suffixes=('_private', '_public'))

merged_df["Rank_difference"] = merged_df["Rank_public"] - merged_df["Rank_private"]

plt.figure(figsize = (10, 8))
plt.style.use("fivethirtyeight")

sns.boxplot(merged_df, x = "Episode_public", y = "Rank_difference", linewidth = 1.5, fliersize = 1, color = "#cc0000")

plt.title("Rank Difference from Public to Private Leaderboard")
plt.text(0, -1200, f"Biggest increase in rank from public to private LB: {merged_df.Rank_difference.max()}", color = "#009933")
plt.text(0, -1300, f"Biggest decrease in rank from public to private LB: {abs(merged_df.Rank_difference.min())}", color = "#cc0000")
plt.xlabel("Episode")
plt.ylabel("Rank Difference")
plt.show()

### Visualizing the number of rows in the training set & testing set in each Episode
- The public LB contains 20% of the testing data and the private LB contains 80% of the testing data for every episode except for Classification with a Tabular Vector Borne Disease Dataset - Episode 13 where the testing data is split 50-50 between the public & private LB.
- Insight: Notice how episodes with small datasets tend to lead to large LB shakeup between the public & private LB. Conversely, episodes with large datasets are less likely to experience a large LB shakeup. Episodes 11 & 12 are good examples of this.

In [ ]:
size_df["Playground Episode"] = [ep for ep in range(1, N_EPISODES + 1)]

fig = px.bar(size_df, x = "Playground Episode", y = ["# Train Rows", "# Test Rows"],
             barmode = "group", color_discrete_sequence = ["#0066ff", "#00ccff"], labels = {"variable": "Dataset"},
             width = 1200, height = 600, template = "plotly_dark",
             title = "Total Number of Rows of Data in the Training & Testing Sets")

fig.update_traces(marker = dict(line = dict(width = 1.5, color = "#FFFFFF")))
fig.update(layout_coloraxis_showscale=False)
fig.update_layout(xaxis = dict(dtick = 1))
fig.show()

### Visualizing the gap between the public & private LB Scores
- Insight: Notice the large amount of overfitting of the public LB in Episode 6

In [ ]:
for ep in range(1, N_EPISODES + 1):
    plt.figure(figsize = (10, 8))
    plt.style.use("fivethirtyeight")
    df0 = df.query(f"Episode == {ep}")
    ranks = len(df0) * 0.35
    sns.lineplot(df0.query("Rank < @ranks"), x = "Rank", y = "Score", hue = "Leaderboard", palette = ["#0066ff", "#cc33ff"], linewidth = 2)
    plt.title(f"LB Score Gap | Episode: {ep} | Evaluation metric: {competition_eval_metrics[ep - 1]}")
    plt.xlabel("Leaderboard Rank")
    plt.show()